# Ollama — LLM local con Python

Ollama permite ejecutar modelos LLM open source directamente en local,
sin API key y sin mandar datos a servidores externos.
Todo ocurre en tu máquina.

En este notebook conectamos Python con Ollama via HTTP,
exactamente igual que con la API de Anthropic / OpenAI —
la diferencia es que el endpoint es `localhost`.

## Qué veremos
- Verificar que Ollama está corriendo y qué modelos tenemos disponibles
- Hacer nuestra primera llamada al modelo desde Python
- Comparar la interfaz con el SDK de Anthropic que ya conocemos


In [ ]:
import httpx

# Verificamos que Ollama está corriendo
response = httpx.get("http://localhost:11434/api/tags")
print(response.json())

## Primera llamada al modelo

Hacemos un POST a la API de Ollama con el mensaje del usuario.
La estructura es idéntica a la API de Anthropic: `model`, `messages` con `role` y `content`.
`stream: False` indica que queremos la respuesta completa de golpe, no token a token.
El resultado llega en `data["message"]["content"]`.

In [ ]:
response = httpx.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "qwen2.5:7b",
        "messages": [{"role": "user", "content": "Explica qué es un LLM en 2 frases"}],
        "stream": False
    },
    timeout=60
)

data = response.json()
print(f"RAW: {data}")
print("\n")
print(data["message"]["content"])

## Analizando la respuesta

La API devuelve metadatos además del texto: duración total, tokens procesados,
tokens generados. Esto es útil para medir rendimiento y coste computacional.
Calculamos los tokens por segundo para benchmarking.

In [ ]:
tokens = data["eval_count"]
duration_s = data["eval_duration"] / 1e9  # nanosegundos a segundos

print(f"Respuesta: {data['message']['content']}")
print(f"\nTokens generados: {tokens}")
print(f"Duración: {duration_s:.2f}s")
print(f"Velocidad: {tokens / duration_s:.1f} tokens/segundo")